# Read-only Fabric / Power BI Semantic Model Diagnostics
**Assessment:** This notebook helps inventory the model and review relationships; it is not the Model Best Practice Analyzer and cannot establish why a report total is wrong. The original optional `list_relationship_violations(MODEL_NAME, workspace=...)` call was invalid: that API accepts loaded FabricDataFrames, not a model name and workspace. The optional check below uses the documented model-level missing-row calculation instead.

**Safety:** No model edits, refreshes, repairs, or lakehouse exports. Metadata-only by default; optional missing-row queries consume capacity and access model data. Results remain in notebook memory/output and may contain sensitive names. Review before sharing.

**Requirements:** Run in a Microsoft Fabric notebook with Semantic Link (`sempy`) installed and authorized access to the target workspace/model. Some TOM-based metadata APIs, including `list_relationships`, require ReadWrite permissions according to the current SemPy documentation even though these calls do not edit the model. Confirm capacity/XMLA availability and tenant settings with your administrator; read-only behavior does not imply a viewer account can run every API.

**Run order:** Set the two names below, run the inventory, then use the companion BPA and memory notebooks as needed. Failures stop execution rather than appearing as empty results.


In [ ]:
WORKSPACE_NAME = ""
MODEL_NAME = ""
RUN_RELATIONSHIP_VIOLATIONS = False

if not WORKSPACE_NAME.strip() or not MODEL_NAME.strip():
    raise ValueError("Set WORKSPACE_NAME and MODEL_NAME to the exact Fabric workspace and semantic model names.")


In [ ]:
import pandas as pd
import sempy.fabric as fabric
from IPython.display import display

def read_inventory(label, func):
    df = func()
    print(f"{label}: {len(df)} records; showing at most 30. Full result retained in its variable.")
    display(df.head(30))
    return df

tables = read_inventory("Tables", lambda: fabric.list_tables(MODEL_NAME, workspace=WORKSPACE_NAME))
columns = read_inventory("Columns", lambda: fabric.list_columns(MODEL_NAME, workspace=WORKSPACE_NAME))
relationships = read_inventory("Relationships", lambda: fabric.list_relationships(MODEL_NAME, workspace=WORKSPACE_NAME))
measures = read_inventory("DAX measures", lambda: fabric.list_measures(MODEL_NAME, workspace=WORKSPACE_NAME))


In [ ]:
if relationships.empty:
    print("The metadata call succeeded and returned no relationships. Verify the selected model and intended design.")
else:
    print("Relationship review: at most 100 rows; complete result is in relationships.")
    print("Available relationship properties:", list(relationships.columns))
    with pd.option_context('display.max_columns', None):
        display(relationships.head(100))
    print("Review endpoints, cardinality, active status, and filter direction together. No automatic defect classification is performed.")
    print("Inactive, many-to-many, and bidirectional relationships can be intentional; validate expected filter paths and business meaning.")


In [ ]:
# Missing-row checks query model data; they are not a complete relationship-validity test.
if RUN_RELATIONSHIP_VIOLATIONS:
    relationship_validation = fabric.list_relationships(
        MODEL_NAME,
        workspace=WORKSPACE_NAME,
        extended=True,
        calculate_missing_rows=True,
    )
    print("Missing-row statistics for all returned relationships (not only violations):")
    with pd.option_context('display.max_columns', None):
        display(relationship_validation)
    print("Inspect missing-row counts. Zero missing rows does not prove correct cardinality, unique keys, or intended filtering.")
else:
    print("Relationship missing-row queries skipped (default). Set RUN_RELATIONSHIP_VIOLATIONS=True to enable.")


## Diagnostic coverage
| Diagnostic | Included / location |
| --- | --- |
| Semantic model tables | This notebook: `tables` |
| Column metadata | This notebook: `columns` |
| Relationships and cardinality | This notebook: `relationships` |
| Active/inactive relationships | This notebook: relationship properties, manual review |
| DAX measures inventory | This notebook: `measures`; does not test measure correctness |
| Relationship violations | Optional missing-row statistics only; not exhaustive key or filter-path validation |
| Best Practice Analyzer | Companion `Fabric Semantic Model Best Practice Analyzer.ipynb`, using Microsoft's Semantic Link Labs API |
| Memory analysis | Companion `Fabric Semantic Model Memory Analysis.ipynb`, using Microsoft's VertiPaq Analyzer API |
| Visual performance timings | Power BI Performance Analyzer, outside these notebooks |

## Interpret findings
- **Tables absent:** compare the complete inventory, not just its preview, with the source and confirm explicit inclusion in the model.
- **Inactive / many-to-many / bidirectional relationships:** review intent; none is automatically a bug. The original notebook displayed properties but did not actually flag defects, and its filtered view omitted the relationship endpoints.
- **Missing rows:** investigate fact keys without matching dimension keys; separately validate dimension-key uniqueness, blanks, types, and ambiguous filter paths.
- **Incorrect KPI:** compare a known-good source aggregation with DAX under the same filters and RLS identity. Neither a clean BPA scan nor a zero missing-row count proves business correctness.
- **Slow visual:** in Power BI open Performance Analyzer, start recording, refresh visuals, and inspect DAX query versus visual-display time. Copy the slow query for further investigation; these notebooks do not time report visuals.

**Execution status:** Authored/reviewed locally; no target workspace/model was supplied and no live diagnostics have been run. Set configuration separately in each notebook. Do not treat this notebook as an analysis result for your actual model.

### References
- [Semantic Link relationships](https://learn.microsoft.com/en-us/fabric/data-science/semantic-link-validate-relationship)
- [Semantic Link Python API: list_relationships and list_relationship_violations](https://learn.microsoft.com/en-us/python/api/semantic-link-sempy/sempy.fabric?view=semantic-link-python)
- [Microsoft Semantic Link Labs examples: BPA and VertiPaq Analyzer](https://github.com/microsoft/semantic-link-labs/wiki/Code-Examples#model-best-practice-analyzer)
- [Performance Analyzer](https://learn.microsoft.com/en-us/power-bi/create-reports/performance-analyzer)
